# Environmental Data Retrieval

This script utilizes **Bacdive** and the **NCBI** to obtain environmental information from various datasets. It integrates data sources to provide comprehensive environmental context.

## Key Features

- Accesses **Bacdive** database for detailed microbial environmental metadata.
- Queries **NCBI** to retrieve sequence and sample data related to environmental samples.
- Combines and filters information to generate an enriched environmental dataset.

> ⚙️ This workflow helps researchers gather robust environmental data alongside sequence information.

---

_Note: Ensure you have access to Bacdive API and NCBI Entrez utilities configured in your environment._


In [ ]:
!pip install bacdive

In [5]:
import pandas as pd
from Bio import Entrez
from tqdm import tqdm
import time
import os
import re

# Set your NCBI email (required)
Entrez.email = "your.email@example.com"  # Replace with your actual email

# === Input and output setup ===
input_csv = "results/sequences/vgsc_min100_max500_cdhit_0.90_taxonomy_lineages.csv"
output_folder = "results/environment"
os.makedirs(output_folder, exist_ok=True)

# Generate base name from input filename
input_filename = os.path.splitext(os.path.basename(input_csv))[0]
if input_filename.endswith("_taxonomy_lineages"):
    base_name = input_filename.replace("_taxonomy_lineages", "")
else:
    base_name = input_filename

# Output filenames
output_csv = os.path.join(output_folder, f"{base_name}_ncbi_source_info.csv")
missing_csv = os.path.join(output_folder, f"{base_name}_ncbi_missing.csv")

# Read input and keep relevant fields
df = pd.read_csv(input_csv)
df = df[['protein_id', 'organism_from_ncb', 'taxonomy_id']].rename(columns={'protein_id': 'protein_id'})
df = df.dropna(subset=['protein_id'])
df['protein_id'] = df['protein_id'].astype(str).str.strip()

# Process all protein IDs
protein_ids = df['protein_id'].tolist()

main_cols = ['organism_from_ncb', 'taxonomy_id', 'strain_id']
dynamic_cols = set()
data_records = []
missing_info_records = []

# === Helper functions ===

def parse_source_features(gb_text):
    source_data = {}
    in_source = False
    for line in gb_text.splitlines():
        line = line.strip()
        if line.startswith("source"):
            in_source = True
        elif in_source and line.startswith("/"):
            match = re.match(r'/(\w+)=["\']?(.+?)["\']?$', line)
            if match:
                key, value = match.groups()
                if key not in ("organism", "db_xref"):
                    source_data[key] = value
        elif in_source and not line.startswith("/"):
            break
    return source_data

def clean_tax_id(tax_str):
    digits = re.findall(r'\d+', tax_str)
    return digits[0] if digits else ""

def validate_protein_ids(protein_ids):
    valid_ids = []
    invalid_ids = []
    for pid in tqdm(protein_ids, desc="Validating protein IDs"):
        for attempt in range(1, 6):
            try:
                handle = Entrez.efetch(db="protein", id=pid, rettype="gb", retmode="text")
                data = handle.read()
                handle.close()
                if data.strip():
                    valid_ids.append(pid)
                else:
                    invalid_ids.append(pid)
                break
            except Exception as e:
                print(f"Attempt {attempt} failed for {pid}: {e}")
                if attempt == 5:
                    invalid_ids.append(pid)
                time.sleep(2 ** attempt)
        time.sleep(0.34)
    return valid_ids, invalid_ids

def parse_gb_record(gb_record):
    organism = None
    tax_id = None
    strain = None
    for line in gb_record.splitlines():
        if line.startswith("  ORGANISM"):
            organism = line.split("  ORGANISM  ")[-1].strip()
        if 'db_xref="taxon:' in line:
            idx = line.find('db_xref="taxon:')
            raw_tax = line[idx+14:idx+14+line[idx+14:].find('"')]
            tax_id = clean_tax_id(raw_tax)
        if '/strain="' in line:
            start = line.find('/strain="') + len('/strain="')
            end = line.find('"', start)
            strain = line[start:end]
    extra_fields = parse_source_features(gb_record)
    return {
        'organism_from_ncb': organism or "",
        'taxonomy_id': tax_id or "",
        'strain_id': strain or "",
        **extra_fields
    }

def batch_fetch_gb_records(protein_ids, batch_size=50):
    records = {}
    for i in tqdm(range(0, len(protein_ids), batch_size), desc="Fetching GenBank records"):
        batch_ids = protein_ids[i:i+batch_size]
        ids_str = ",".join(batch_ids)
        for attempt in range(1, 6):
            try:
                handle = Entrez.efetch(db="protein", id=ids_str, rettype="gb", retmode="text")
                data = handle.read()
                handle.close()
                gb_recs = data.strip().split("\n//\n")
                for idx_rec, rec in enumerate(gb_recs):
                    pid = batch_ids[idx_rec]
                    records[pid] = rec
                break
            except Exception as e:
                print(f"Attempt {attempt} failed efetch batch {ids_str}: {e}")
                if attempt == 5:
                    print(f"Giving up batch {ids_str} after 5 tries.")
                time.sleep(2 ** attempt)
        time.sleep(0.34)
    return records

# === Main processing ===

valid_ids, invalid_ids = validate_protein_ids(protein_ids)
gb_records = batch_fetch_gb_records(valid_ids, batch_size=50)

missing_ids = []

for protein_id in tqdm(protein_ids, desc="Parsing records"):
    if protein_id in gb_records:
        info = parse_gb_record(gb_records[protein_id])
    else:
        info = {'organism_from_ncb': "", 'taxonomy_id': "", 'strain_id': ""}

    is_missing = not info['taxonomy_id'] or not info['organism_from_ncb']
    new_keys = set(info.keys()) - set(main_cols)
    dynamic_cols.update(new_keys)

    record = {'protein_id': protein_id}
    for col in main_cols:
        record[col] = info.get(col, "")
    for key in new_keys:
        record[key] = info.get(key, "")

    if is_missing:
        missing_info_records.append(record)
        missing_ids.append(protein_id)
    else:
        data_records.append(record)

# === Output consolidated files ===

final_cols = ['protein_id'] + main_cols + sorted(dynamic_cols)

output_df = pd.DataFrame(data_records, columns=final_cols)
output_df.to_csv(output_csv, index=False)
print(f"\n✅ Consolidated output saved to: {output_csv} ({len(data_records)} records)")

if missing_info_records:
    missing_df = pd.DataFrame(missing_info_records, columns=final_cols)
    missing_df.to_csv(missing_csv, index=False)
    print(f"⚠️ Missing info saved to: {missing_csv} ({len(missing_info_records)} records)")

# === Summary reporting ===

total = len(protein_ids)
successes = len(data_records)
failures = len(missing_info_records)

missing_tax_id = sum(1 for rec in missing_info_records if not rec.get('taxonomy_id'))
missing_organism = sum(1 for rec in missing_info_records if not rec.get('organism_from_ncb'))

print("\n📊 Summary Report")
print(f"  Total protein IDs processed: {total}")
print(f"  ✅ Successful NCBI fetches   : {successes}/{total}")
print(f"  ⚠️ Missing records           : {failures}/{total}")
print(f"    - Missing taxonomy_id      : {missing_tax_id}/{total}")
print(f"    - Missing organism name    : {missing_organism}/{total}")
print(f"  ✅ All accounted for         : {successes + failures == total}")

if missing_ids:
    print("\n⚠️ Missing protein IDs:")
    for pid in missing_ids:
        print(f"  - {pid}")
else:
    print("\n✅ No missing protein IDs.")


Parsing records: 100%|█████████████████████████████████████████████████████████████| 828/828 [00:00<00:00, 6080.02it/s]


✅ Consolidated output saved to: results/environment\vgsc_min100_max500_cdhit_0.90_ncbi_source_info.csv (828 records)

📊 Summary Report
  Total protein IDs processed: 828
  ✅ Successful NCBI fetches   : 828/828
  ⚠️ Missing records           : 0/828
    - Missing taxonomy_id      : 0/828
    - Missing organism name    : 0/828
  ✅ All accounted for         : True

✅ No missing protein IDs.


In [7]:
import pandas as pd

# Replace 'your_file.csv' with the path to your CSV file
csv_file = 'results/environment/vgsc_min100_max500_cdhit_0.90_ncbi_source_info.csv'

# Read the CSV file
df = pd.read_csv(csv_file)

# Print the column names
print("Column names:")
for column in df.columns:
    print(column)


Column names:
protein_id
organism_from_ncb
taxonomy_id
strain_id
altitude
chromosome
collected_by
collection_date
culture_collection
geo_loc_name
host
identified_by
isolate
isolation_source
lab_host
lat_lon
metagenome_source
note
plasmid
specimen_voucher
strain
sub_species
type_material


In [11]:
import pandas as pd
import os

# === File paths ===
input_path = 'results/environment/vgsc_min100_max500_cdhit_0.90_ncbi_source_info.csv'
#missing_path = 'results/environment/vgsc-id90_min100_max500_ncbi_missing.csv'  # Optional

columns_to_delete = [
    'altitude', 
    'chromosome', 
    'collected_by', 
    'collection_date',
    'identified_by',
    'lab_host',
    'lat_lon', 
    'metagenome_source',
    'note',
    'plasmid',
    'specimen_voucher', 
    'strain', 
    'sub_species'
]

# === Cleaning function ===
def clean_protein_ids(series):
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r'\s+', '', regex=True)  # Remove all whitespace (spaces, tabs, newlines)
    )

# === Load input CSV ===
df = pd.read_csv(input_path)

if 'protein_id' not in df.columns:
    raise ValueError("❌ 'protein_id' column not found in source file.")

df['protein_id'] = clean_protein_ids(df['protein_id'])

# === Remove missing protein_ids ===
if missing_path and os.path.exists(missing_path):
    missing_df = pd.read_csv(missing_path)

    if 'protein_id' not in missing_df.columns:
        raise ValueError("❌ 'protein_id' column not found in missing file.")
    
    missing_df['protein_id'] = clean_protein_ids(missing_df['protein_id'])
    missing_ids = set(missing_df['protein_id'])
    
    all_ids = set(df['protein_id'])
    matched_ids = missing_ids.intersection(all_ids)

    original_count = len(df)
    df = df[~df['protein_id'].isin(matched_ids)]
    removed_count = original_count - len(df)

    if removed_count > 0:
        print(f"🧹 Removed {removed_count} matching entries from missing file ({len(matched_ids)} matches)")
    else:
        print("⚠️ No matching protein_id entries found to remove.")
else:
    print(f"⚠️ Skipping missing file: {missing_path if missing_path else 'Not provided'}")

# === Drop unwanted columns (safely) ===
df_cleaned = df.drop(columns=[col for col in columns_to_delete if col in df.columns])

# === Normalize taxonomy_id ===
df_cleaned['taxonomy_id'] = df_cleaned['taxonomy_id'].fillna(0).astype(int).astype(str)

# === Save cleaned output ===
base, ext = os.path.splitext(input_path)
output_path = f"{base}_cleaned{ext}"

df_cleaned.to_csv(output_path, index=False)
print(f"✅ Done. Cleaned CSV saved to: {output_path}")


⚠️ Skipping missing file: results/environment/vgsc-id90_min100_max500_ncbi_missing.csv
✅ Done. Cleaned CSV saved to: results/environment/vgsc_min100_max500_cdhit_0.90_ncbi_source_info_cleaned.csv


now lets try bacdive; it worked!

In [2]:
#this is an improve version of the code 
import pandas as pd
import os
import sys
import io
from bacdive import BacdiveClient
from tqdm import tqdm

# === Toggle Test Mode ===
TEST_MODE = False  # Set to False to process the full dataset

# Manual BacDive IDs to test
manual_bacdive_ids = [4301]

# Load input CSV
input_csv = "results/environment/vgsc_min100_max500_cdhit_0.90_ncbi_source_info_cleaned.csv"
df = pd.read_csv(input_csv)

# Extract base name
base_filename = os.path.basename(input_csv)
base_name = base_filename.split('_ncbi')[0]

# Output filenames (add "_test" if test mode is on)
suffix = "_test" if TEST_MODE else ""
output_file = f"results/environment/{base_name}_bacdive_source_info_found{suffix}.csv"
not_found_file = f"results/environment/{base_name}_bacdive_source_info_missing{suffix}.csv"

# Initialize BacDive client
client = BacdiveClient('username', 'password') # insert username and password

# Results containers
results = []
not_found = []
processed_taxonomy_ids = set()

# === Silent wrapper to suppress stdout ===
def silent_search(func, *args, **kwargs):
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        return func(*args, **kwargs)
    finally:
        sys.stdout = old_stdout

# === Step 1: Retrieve manual BacDive IDs ===
if TEST_MODE and manual_bacdive_ids:
    print(f"🔬 Fetching manual BacDive ID(s): {manual_bacdive_ids}")
    client.setSearchType('exact')
    silent_search(client.search, id=manual_bacdive_ids)
    for strain_data in client.retrieve():
        bacdive_id = strain_data.get('General', {}).get('BacDive-ID')
        if not bacdive_id:
            continue

        general = strain_data.get('General', {})
        description = general.get('description', '')
        keywords = general.get('keywords', '')
        if isinstance(keywords, list):
            keywords = ", ".join(keywords)

        isolation = strain_data.get('Isolation, sampling and environmental information', {})
        iso_cat = isolation.get('isolation source categories', [])
        iso_cat_str = ''
        if isinstance(iso_cat, list):
            category_values = []
            for cat_dict in iso_cat:
                if isinstance(cat_dict, dict):
                    category_values.extend([v for v in cat_dict.values() if v])
            iso_cat_str = ", ".join(sorted(set(category_values)))

        results.append({
            "organism_from_ncb": '',
            "taxonomy_id": '',
            "strain_id": '',
            "BacDive-ID": bacdive_id,
            "Description": description,
            "Keywords": keywords,
            "Growth Temp": '',
            "Optimum Temp": '',
            "pH Growth": '',
            "pH Optimum": '',
            "pH Range": '',
            "Isolation Sample Type": '',
            "Isolation Geographic Location": '',
            "Isolation Country": '',
            "Isolation Continent": '',
            "Isolation Source Categories": iso_cat_str,
            "Total Samples": '',
            "Soil Counts": '',
            "Aquatic Counts": '',
            "Animal Counts": '',
            "Plant Counts": ''
        })

# === Step 2: Process first 10 rows from CSV (test) or full (normal) ===
rows_to_process = df.head(10) if TEST_MODE else df

# Loop through rows
for _, row in tqdm(rows_to_process.iterrows(), total=len(rows_to_process), desc="Fetching BacDive entries"):
    strain_id = str(row.get("strain_id", "")).strip()
    organism = str(row.get("organism_from_ncb", "")).strip()
    tax_id = str(row.get("taxonomy_id", "")).strip()

    if tax_id in processed_taxonomy_ids:
        continue

    data_found = False
    bacdive_ids_added = set()

    count = 0
    if strain_id:
        client.setSearchType('exact')
        count = silent_search(client.search, culturecolno=strain_id)

    if count == 0 and organism:
        client.setSearchType('startswith')
        count = silent_search(client.search, taxonomy=organism)

    if count == 0:
        not_found.append({
            "organism_from_ncb": organism,
            "taxonomy_id": tax_id,
            "strain_id": strain_id
        })
        processed_taxonomy_ids.add(tax_id)
        continue

    for strain_data in client.retrieve():
        bacdive_id = strain_data.get('General', {}).get('BacDive-ID')
        if not bacdive_id or bacdive_id in bacdive_ids_added:
            continue

        bacdive_ids_added.add(bacdive_id)
        data_found = True

        general = strain_data.get('General', {})
        description = general.get('description', '')
        keywords = general.get('keywords', '')
        if isinstance(keywords, list):
            keywords = ", ".join(keywords)

        # Temperature
        temps = strain_data.get('Culture and growth conditions', {}).get('culture temp', [])
        growth_temps = set()
        optimum_temps = set()
        for entry in temps:
            if isinstance(entry, dict):
                if entry.get('type') == 'growth':
                    growth_temps.add(entry.get('temperature', ''))
                elif entry.get('type') == 'optimum':
                    optimum_temps.add(entry.get('temperature', ''))

        # pH
        ph_entries = strain_data.get('Culture and growth conditions', {}).get('culture pH', [])
        ph_growth = ''
        ph_optimum = ''
        ph_range = ''
        for entry in ph_entries:
            if isinstance(entry, dict):
                if entry.get('type') == 'growth':
                    ph_growth = entry.get('pH', '')
                    ph_range = entry.get('PH range', '')
                elif entry.get('type') == 'optimum':
                    ph_optimum = entry.get('pH', '')

        # Isolation
        isolation = strain_data.get('Isolation, sampling and environmental information', {})
        iso_entries = isolation.get('isolation', [])
        sample_type = geo_location = country = continent = ''
        if isinstance(iso_entries, list) and iso_entries:
            first = iso_entries[0]
            sample_type = first.get('sample type', '')
            geo_location = first.get('geographic location', '')
            country = first.get('country', '')
            continent = first.get('continent', '')
        elif isinstance(iso_entries, dict):
            sample_type = iso_entries.get('sample type', '')
            geo_location = iso_entries.get('geographic location', '')
            country = iso_entries.get('country', '')
            continent = iso_entries.get('continent', '')

        # Source categories
        iso_cat = isolation.get('isolation source categories', [])
        iso_cat_str = ''
        if isinstance(iso_cat, list):
            category_values = []
            for cat_dict in iso_cat:
                if isinstance(cat_dict, dict):
                    category_values.extend([v for v in cat_dict.values() if v])
            iso_cat_str = ", ".join(sorted(set(category_values)))

        taxonmaps = isolation.get('taxonmaps', {})
        total_samples = taxonmaps.get('Total samples', '')
        soil_counts = taxonmaps.get('soil counts', '')
        aquatic_counts = taxonmaps.get('aquatic counts', '')
        animal_counts = taxonmaps.get('animal counts', '')
        plant_counts = taxonmaps.get('plant counts', '')

        results.append({
            "organism_from_ncb": organism,
            "taxonomy_id": tax_id,
            "strain_id": strain_id,
            "BacDive-ID": bacdive_id,
            "Description": description,
            "Keywords": keywords,
            "Growth Temp": ", ".join(sorted(growth_temps)),
            "Optimum Temp": ", ".join(sorted(optimum_temps)),
            "pH Growth": ph_growth,
            "pH Optimum": ph_optimum,
            "pH Range": ph_range,
            "Isolation Sample Type": sample_type,
            "Isolation Geographic Location": geo_location,
            "Isolation Country": country,
            "Isolation Continent": continent,
            "Isolation Source Categories": iso_cat_str,
            "Total Samples": total_samples,
            "Soil Counts": soil_counts,
            "Aquatic Counts": aquatic_counts,
            "Animal Counts": animal_counts,
            "Plant Counts": plant_counts
        })

    processed_taxonomy_ids.add(tax_id)

    if not data_found:
        not_found.append({
            "organism_from_ncb": organism,
            "taxonomy_id": tax_id,
            "strain_id": strain_id
        })

# Save CSVs
pd.DataFrame(results).to_csv(output_file, index=False)
pd.DataFrame(not_found).to_csv(not_found_file, index=False)

print(f"\n✅ Saved: {output_file}")
print(f"❌ Missing: {not_found_file}")


-- Authentication successful --


Fetching BacDive entries: 100%|██████████████████████████████████████████████████████| 828/828 [23:26<00:00,  1.70s/it]


✅ Saved: results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found.csv
❌ Missing: results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_missing.csv


In [ ]:
#ignore the code below, rather use the code above 

In [13]:
# import pandas as pd
# import os
# import sys
# import io
# from bacdive import BacdiveClient
# from tqdm import tqdm

# # Load data
# input_csv = "results/environment/vgsc_min100_max500_cdhit_0.90_ncbi_source_info_cleaned.csv"
# df = pd.read_csv(input_csv)

# # Extract base name
# base_filename = os.path.basename(input_csv)
# base_name = base_filename.split('_ncbi')[0]

# # Output filenames
# output_file = f"results/environment/{base_name}_bacdive_source_info_found.csv"
# not_found_file = f"results/environment/{base_name}_bacdive_source_info_missing.csv"

# # Initialize BacDive client
# client = BacdiveClient('username', 'password
')

# fieldnames = [
#     "organism_from_ncb", "taxonomy_id", "strain_id",
#     "BacDive-ID", "Description", "Keywords",
#     "Growth Temp", "Optimum Temp", "pH Growth", "pH Optimum", "pH Range",
#     "Isolation Sample Type", "Isolation Geographic Location", "Isolation Country", "Isolation Continent",
#     "Isolation Source Categories", "Total Samples", "Soil Counts", "Aquatic Counts", "Animal Counts", "Plant Counts"
# ]

# results = []
# not_found = []

# # Track processed taxonomy IDs
# processed_taxonomy_ids = set()

# def silent_search(func, *args, **kwargs):
#     """Suppress stdout during BacDive search."""
#     old_stdout = sys.stdout
#     sys.stdout = io.StringIO()
#     try:
#         return func(*args, **kwargs)
#     finally:
#         sys.stdout = old_stdout

# # Loop with progress bar
# for _, row in tqdm(df.iterrows(), total=len(df), desc="Fetching BacDive entries"):
#     strain_id = str(row.get("strain_id", "")).strip()
#     organism = str(row.get("organism_from_ncb", "")).strip()
#     tax_id = str(row.get("taxonomy_id", "")).strip()

#     if tax_id in processed_taxonomy_ids:
#         continue

#     data_found = False
#     bacdive_ids_added = set()

#     # Try culture collection number
#     count = 0
#     if strain_id:
#         client.setSearchType('exact')
#         count = silent_search(client.search, culturecolno=strain_id)

#     # If no result, try taxonomy name
#     if count == 0 and organism:
#         client.setSearchType('startswith')
#         count = silent_search(client.search, taxonomy=organism)

#     # If still nothing, log as not found
#     if count == 0:
#         not_found.append({
#             "organism_from_ncb": organism,
#             "taxonomy_id": tax_id,
#             "strain_id": strain_id
#         })
#         processed_taxonomy_ids.add(tax_id)
#         continue

#     for strain_data in client.retrieve():
#         bacdive_id = strain_data.get('General', {}).get('BacDive-ID')
#         if not bacdive_id or bacdive_id in bacdive_ids_added:
#             continue

#         bacdive_ids_added.add(bacdive_id)
#         data_found = True

#         general = strain_data.get('General', {})
#         description = general.get('description', '')
#         keywords = general.get('keywords', '')
#         if isinstance(keywords, list):
#             keywords = ", ".join(keywords)

#         # Temperature
#         temps = strain_data.get('Culture and growth conditions', {}).get('culture temp', [])
#         growth_temps = set()
#         optimum_temps = set()
#         for entry in temps:
#             if isinstance(entry, dict):
#                 if entry.get('type') == 'growth':
#                     growth_temps.add(entry.get('temperature', ''))
#                 elif entry.get('type') == 'optimum':
#                     optimum_temps.add(entry.get('temperature', ''))


#         # pH
#         ph_entries = strain_data.get('Culture and growth conditions', {}).get('culture pH', [])
#         ph_growth = ''
#         ph_optimum = ''
#         ph_range = ''
#         for entry in ph_entries:
#             if isinstance(entry, dict):
#                 if entry.get('type') == 'growth':
#                     ph_growth = entry.get('pH', '')
#                     ph_range = entry.get('PH range', '')
#                 elif entry.get('type') == 'optimum':
#                     ph_optimum = entry.get('pH', '')


#         # Isolation
#         isolation = strain_data.get('Isolation, sampling and environmental information', {})
#         iso_entries = isolation.get('isolation', [])
#         sample_type = geo_location = country = continent = ''

#         if isinstance(iso_entries, list) and iso_entries:
#             first = iso_entries[0]
#             sample_type = first.get('sample type', '')
#             geo_location = first.get('geographic location', '')
#             country = first.get('country', '')
#             continent = first.get('continent', '')
#         elif isinstance(iso_entries, dict):
#             sample_type = iso_entries.get('sample type', '')
#             geo_location = iso_entries.get('geographic location', '')
#             country = iso_entries.get('country', '')
#             continent = iso_entries.get('continent', '')

#         iso_cat = isolation.get('isolation source categories', {})
#         if isinstance(iso_cat, dict):
#             iso_cat_str = ", ".join(iso_cat.values())
#         else:
#             iso_cat_str = ''

#         taxonmaps = isolation.get('taxonmaps', {})
#         total_samples = taxonmaps.get('Total samples', '')
#         soil_counts = taxonmaps.get('soil counts', '')
#         aquatic_counts = taxonmaps.get('aquatic counts', '')
#         animal_counts = taxonmaps.get('animal counts', '')
#         plant_counts = taxonmaps.get('plant counts', '')

#         results.append({
#             "organism_from_ncb": organism,
#             "taxonomy_id": tax_id,
#             "strain_id": strain_id,
#             "BacDive-ID": bacdive_id,
#             "Description": description,
#             "Keywords": keywords,
#             "Growth Temp": ", ".join(sorted(growth_temps)),
#             "Optimum Temp": ", ".join(sorted(optimum_temps)),
#             "pH Growth": ph_growth,
#             "pH Optimum": ph_optimum,
#             "pH Range": ph_range,
#             "Isolation Sample Type": sample_type,
#             "Isolation Geographic Location": geo_location,
#             "Isolation Country": country,
#             "Isolation Continent": continent,
#             "Isolation Source Categories": iso_cat_str,
#             "Total Samples": total_samples,
#             "Soil Counts": soil_counts,
#             "Aquatic Counts": aquatic_counts,
#             "Animal Counts": animal_counts,
#             "Plant Counts": plant_counts
#         })

#     processed_taxonomy_ids.add(tax_id)

#     if not data_found:
#         not_found.append({
#             "organism_from_ncb": organism,
#             "taxonomy_id": tax_id,
#             "strain_id": strain_id
#         })

# # Save CSVs
# pd.DataFrame(results).to_csv(output_file, index=False)
# pd.DataFrame(not_found).to_csv(not_found_file, index=False)

# print(f"\n✅ Saved: {output_file}")
# print(f"❌ Missing: {not_found_file}")


-- Authentication successful --


Fetching BacDive entries: 100%|██████████████████████████████████████████████████████| 828/828 [19:31<00:00,  1.41s/it]


✅ Saved: results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found.csv
❌ Missing: results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_missing.csv


data aquisition is done

In [3]:
import pandas as pd

# Define file paths
file_missing = "results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_missing.csv"
file_found = "results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found.csv"
file_reference = "results/environment/vgsc_min100_max500_cdhit_0.90_ncbi_source_info_cleaned.csv"

# Read CSV files
df_missing = pd.read_csv(file_missing)
df_found = pd.read_csv(file_found)
df_reference = pd.read_csv(file_reference)

# Get unique taxonomy_ids from each file
unique_missing_ids = df_missing['taxonomy_id'].dropna().drop_duplicates()
unique_found_ids = df_found['taxonomy_id'].dropna().drop_duplicates()
unique_reference_ids = df_reference['taxonomy_id'].dropna().drop_duplicates()

# Combine and get total unique taxonomy_ids from missing + found
combined_unique_ids = pd.concat([unique_missing_ids, unique_found_ids]).drop_duplicates()

# Print results
print("Number of unique taxonomy_id in 'missing' file:", len(unique_missing_ids))
print("Number of unique taxonomy_id in 'found' file:", len(unique_found_ids))
print("Total number of unique taxonomy_id across 'missing' and 'found' files:", len(combined_unique_ids))
print("Number of unique taxonomy_id in reference file:", len(unique_reference_ids))

# Check if they match
if len(combined_unique_ids) == len(unique_reference_ids):
    print("✅ The combined unique taxonomy_ids match the reference file.")
else:
    print("❌ Mismatch detected.")
    print("Difference in count:", len(unique_reference_ids) - len(combined_unique_ids))

    # Show which taxonomy IDs are missing
    missing_in_combined = unique_reference_ids[~unique_reference_ids.isin(combined_unique_ids)]
    if not missing_in_combined.empty:
        print("\nTaxonomy IDs in reference but missing from 'missing' + 'found' files:")
        print(missing_in_combined.tolist())


Number of unique taxonomy_id in 'missing' file: 215
Number of unique taxonomy_id in 'found' file: 546
Total number of unique taxonomy_id across 'missing' and 'found' files: 761
Number of unique taxonomy_id in reference file: 761
✅ The combined unique taxonomy_ids match the reference file.


In [4]:
import pandas as pd

# Replace 'your_file.csv' with the path to your CSV file
csv_file = 'results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found.csv'

# Read the CSV file
df = pd.read_csv(csv_file)

# Print the column names
print("Column names:")
for column in df.columns:
    print(column)


Column names:
organism_from_ncb
taxonomy_id
strain_id
BacDive-ID
Description
Keywords
Growth Temp
Optimum Temp
pH Growth
pH Optimum
pH Range
Isolation Sample Type
Isolation Geographic Location
Isolation Country
Isolation Continent
Isolation Source Categories
Total Samples
Soil Counts
Aquatic Counts
Animal Counts
Plant Counts


--cleaning up data

In [6]:
import pandas as pd
from difflib import SequenceMatcher

# Path to the input file (the one being cleaned)
input_path = "results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found.csv"

# Load data
df = pd.read_csv(input_path, dtype=str)

# Clean whitespace
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Save original unique taxonomy_id count before cleaning
original_tax_ids = df['taxonomy_id'].nunique()

# Columns to group by (ignore BacDive-ID and Description)
columns_to_check = [col for col in df.columns if col not in ['BacDive-ID', 'Description']]

# Track rows to keep
rows_to_keep = []

# Group by all identical fields except BacDive-ID and Description
grouped = df.groupby(columns_to_check, dropna=False)

# Similarity threshold for description (adjust if needed)
SIMILARITY_THRESHOLD = 0.9

for _, group in grouped:
    if len(group) == 1:
        rows_to_keep.append(group.index[0])
    else:
        # Compare descriptions within the group
        descriptions = group['Description'].fillna("")
        kept = set()
        used = [False] * len(group)

        for i in range(len(group)):
            if used[i]:
                continue
            desc_i = descriptions.iloc[i]
            index_i = descriptions.index[i]

            for j in range(i + 1, len(group)):
                if used[j]:
                    continue
                desc_j = descriptions.iloc[j]
                similarity = SequenceMatcher(None, desc_i, desc_j).ratio()

                if similarity >= SIMILARITY_THRESHOLD:
                    # Mark this duplicate for removal (keep only one)
                    used[j] = True

            used[i] = True
            kept.add(index_i)

        rows_to_keep.extend(list(kept))

# Filter the DataFrame to keep only the selected rows
df_cleaned = df.loc[rows_to_keep]

# Count eliminated rows
eliminated = len(df) - len(df_cleaned)
print(f"Rows eliminated: {eliminated}")

# Check if unique taxonomy_id count is the same as the original input
cleaned_tax_ids = df_cleaned['taxonomy_id'].nunique()
same_tax_ids = cleaned_tax_ids == original_tax_ids

print(f"Unique taxonomy_id count before cleaning: {original_tax_ids}")
print(f"Unique taxonomy_id count after cleaning: {cleaned_tax_ids}")
print(f"Do unique taxonomy_id counts match? {same_tax_ids}")

# Save cleaned file
output_path = input_path.replace(".csv", "_cleaned.csv")
df_cleaned.to_csv(output_path, index=False)
print(f"Cleaned file saved to: {output_path}")


Rows eliminated: 326
Unique taxonomy_id count before cleaning: 546
Unique taxonomy_id count after cleaning: 546
Do unique taxonomy_id counts match? True
Cleaned file saved to: results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found_cleaned.csv


In [ ]:
this will only take environmental stats

In [11]:
import pandas as pd
import os

# Define the input path
input_path = "results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found_cleaned.csv"

# Load the CSV file
df = pd.read_csv(input_path)

# Filter rows where 'Total Samples' is not null/empty
filtered_df = df[df["Total Samples"].notna() & (df["Total Samples"] != "")]

# Generate output path
base, ext = os.path.splitext(input_path)
output_path = f"{base}_environmental_distribution{ext}"

# Save the filtered DataFrame to the new CSV
filtered_df.to_csv(output_path, index=False)

print(f"Filtered file saved to: {output_path}")


Filtered file saved to: results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found_cleaned_environmental_distribution.csv


In [17]:
import pandas as pd

# === 1. Load input file ===
input_path = "results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found_cleaned_environmental_distribution.csv"
df = pd.read_csv(input_path)

# === 2. Rename columns with underscores ===
df.rename(columns={
    "organism_from_ncb": "organism_from_ncb",
    "taxonomy_id": "taxonomy_id",
    "strain_id": "strain_id",
    "BacDive-ID": "bacdive_id",
    "Description": "description",
    "Total Samples": "total_samples",
    "Soil Counts": "soil_counts",
    "Aquatic Counts": "aquatic_counts",
    "Animal Counts": "animal_counts",
    "Plant Counts": "plant_counts"
}, inplace=True)

# === 3. Ensure numeric values for environment counts ===
env_cols = ["soil_counts", "aquatic_counts", "animal_counts", "plant_counts"]
df[env_cols] = df[env_cols].fillna(0).apply(pd.to_numeric, errors='coerce').fillna(0)
df["total_samples"] = pd.to_numeric(df["total_samples"], errors='coerce').fillna(0)

# === 4. Classify environment ===
def classify_environment(row):
    total = row["total_samples"]
    if total == 0:
        return pd.Series(["unclassified", "no_data"])
    
    percentages = {
        "soil": row["soil_counts"] / total * 100,
        "aquatic": row["aquatic_counts"] / total * 100,
        "animal": row["animal_counts"] / total * 100,
        "plant": row["plant_counts"] / total * 100
    }

    primary_env = [env for env, pct in percentages.items() if pct >= 60]

    if len(primary_env) == 1:
        classification = primary_env[0]
        detail = f"{percentages[primary_env[0]]:.0f}% {primary_env[0]}"
    else:
        classification = "multienvironmental"
        detail_parts = [f"{pct:.0f}% {env}" for env, pct in percentages.items() if pct > 0]
        detail = ", ".join(detail_parts) if detail_parts else "no_data"
    
    return pd.Series([classification, detail])

df[["environmental_classification", "environment_distribution_details"]] = df.apply(classify_environment, axis=1)

# === 5. Create output DataFrame including count columns for conflict checking ===
output_columns = [
    "organism_from_ncb", "taxonomy_id", "strain_id", "bacdive_id", "description",
    "total_samples", "soil_counts", "aquatic_counts", "animal_counts", "plant_counts",
    "environmental_classification", "environment_distribution_details"
]
output_df = df[output_columns]

# === 6. Save to new CSV ===
output_path = input_path.replace(".csv", "_environmental_classification.csv")
output_df.to_csv(output_path, index=False)
print(f"\n✅ Environmental classification file saved to: {output_path}")

# === 7. Summary statistics ===
unique_species = output_df["taxonomy_id"].nunique()
print(f"\n📊 Total unique species (by taxonomy_id): {unique_species}")

# Unique species per classification
classification_summary = output_df.groupby("taxonomy_id")["environmental_classification"].agg(lambda x: ','.join(sorted(set(x)))).reset_index()
classification_counts = classification_summary["environmental_classification"].value_counts()

print("\n📚 Number of unique species per environmental classification:")
for cls, count in classification_counts.items():
    print(f"  {cls}: {count}")

# === 8. Detect taxonomy_ids with conflicting classifications ===
conflicting_tax_ids = classification_summary[classification_summary["environmental_classification"].str.contains(",")]

if not conflicting_tax_ids.empty:
    print("\n⚠️ Conflicting environmental classifications found for the following taxonomy_ids:")
    for _, row in conflicting_tax_ids.iterrows():
        print(f"  taxonomy_id: {row['taxonomy_id']} → classifications: {row['environmental_classification']}")
else:
    print("\n✅ No conflicting environmental classifications found for any taxonomy_id.")

# === 9. Check repeated taxonomy_ids for identical or inconsistent counts ===
repeated_tax_ids = output_df.groupby("taxonomy_id").filter(lambda x: len(x) > 1)
grouped = repeated_tax_ids.groupby("taxonomy_id")

print("\n🔍 Checking taxonomy_ids that appear multiple times:")

for tax_id, group in grouped:
    class_unique = group["environmental_classification"].nunique()
    dist_unique = group["environment_distribution_details"].nunique()
    env_counts_unique = group[["soil_counts", "aquatic_counts", "animal_counts", "plant_counts"]].drop_duplicates().shape[0]

    if class_unique == 1 and dist_unique == 1 and env_counts_unique == 1:
        print(f"  ✅ taxonomy_id {tax_id} appears multiple times with same environmental distribution — likely same sample with different BacDive IDs.")
    elif env_counts_unique > 1:
        print(f"  ⚠️ taxonomy_id {tax_id} appears multiple times with different environmental count values — check for data consistency.")



✅ Environmental classification file saved to: results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found_cleaned_environmental_distribution_environmental_classification.csv

📊 Total unique species (by taxonomy_id): 392

📚 Number of unique species per environmental classification:
  aquatic: 220
  multienvironmental: 113
  soil: 41
  animal: 13
  plant: 3
  aquatic,multienvironmental: 1
  animal,multienvironmental: 1

⚠️ Conflicting environmental classifications found for the following taxonomy_ids:
  taxonomy_id: 672 → classifications: aquatic,multienvironmental
  taxonomy_id: 1886 → classifications: animal,multienvironmental

🔍 Checking taxonomy_ids that appear multiple times:
  ⚠️ taxonomy_id 301 appears multiple times with different environmental count values — check for data consistency.
  ⚠️ taxonomy_id 672 appears multiple times with different environmental count values — check for data consistency.
  ✅ taxonomy_id 1048 appears multiple times with same environme

In [18]:
# === Recalculate classification from summed counts for inconsistent taxonomy_ids ===

def recalculate_classification(row):
    total = row["total_samples"]
    if total == 0:
        return "unclassified", "no_data"
    
    percentages = {
        "soil": row["soil_counts"] / total * 100,
        "aquatic": row["aquatic_counts"] / total * 100,
        "animal": row["animal_counts"] / total * 100,
        "plant": row["plant_counts"] / total * 100
    }

    primary_env = [env for env, pct in percentages.items() if pct >= 60]

    if len(primary_env) == 1:
        classification = primary_env[0]
        detail = f"{percentages[primary_env[0]]:.0f}% {primary_env[0]}"
    else:
        classification = "multienvironmental"
        detail_parts = [f"{pct:.0f}% {env}" for env, pct in percentages.items() if pct > 0]
        detail = ", ".join(detail_parts) if detail_parts else "no_data"
    
    return classification, detail

# Step 1: Identify taxonomy_ids with inconsistent environmental counts
inconsistent_tax_ids = []

for tax_id, group in grouped:
    env_counts_unique = group[["soil_counts", "aquatic_counts", "animal_counts", "plant_counts"]].drop_duplicates().shape[0]
    if env_counts_unique > 1:
        inconsistent_tax_ids.append(tax_id)

print("\n🔬 Reclassifying taxonomy_ids with inconsistent environmental counts:")

# Step 2: Sum counts and recompute classification
for tax_id in inconsistent_tax_ids:
    group = output_df[output_df["taxonomy_id"] == tax_id]
    summed = group[["soil_counts", "aquatic_counts", "animal_counts", "plant_counts", "total_samples"]].sum()
    
    new_classification, new_detail = recalculate_classification(summed)
    
    # Get unique existing classifications from the group
    original_classes = group["environmental_classification"].unique()
    
    if new_classification not in original_classes or len(original_classes) > 1:
        print(f"\n⚠️ taxonomy_id {tax_id}:")
        print(f"  Original classifications: {', '.join(original_classes)}")
        print(f"  Recalculated classification: {new_classification} ({new_detail})")



🔬 Reclassifying taxonomy_ids with inconsistent environmental counts:

⚠️ taxonomy_id 672:
  Original classifications: multienvironmental, aquatic
  Recalculated classification: multienvironmental (4% soil, 59% aquatic, 36% animal, 1% plant)

⚠️ taxonomy_id 1886:
  Original classifications: animal, multienvironmental
  Recalculated classification: multienvironmental (39% soil, 15% aquatic, 43% animal, 4% plant)


In [22]:
import pandas as pd

# === Load input file ===
input_path = "results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found_cleaned_environmental_distribution.csv"
df = pd.read_csv(input_path)

# === Rename columns with underscores ===
df.rename(columns={
    "organism_from_ncb": "organism_from_ncb",
    "taxonomy_id": "taxonomy_id",
    "strain_id": "strain_id",
    "BacDive-ID": "bacdive_id",
    "Description": "description",
    "Total Samples": "total_samples",
    "Soil Counts": "soil_counts",
    "Aquatic Counts": "aquatic_counts",
    "Animal Counts": "animal_counts",
    "Plant Counts": "plant_counts"
}, inplace=True)

# === Ensure count columns are numeric ===
env_cols = ["soil_counts", "aquatic_counts", "animal_counts", "plant_counts"]
df[env_cols] = df[env_cols].fillna(0).apply(pd.to_numeric, errors='coerce').fillna(0)
df["total_samples"] = pd.to_numeric(df["total_samples"], errors='coerce').fillna(0)

# === Function to classify environment ===
def classify_environment(row):
    total = row["total_samples"]
    if total == 0:
        return "unclassified", "no_data"

    percentages = {
        "soil": row["soil_counts"] / total * 100,
        "aquatic": row["aquatic_counts"] / total * 100,
        "animal": row["animal_counts"] / total * 100,
        "plant": row["plant_counts"] / total * 100
    }

    primary_env = [env for env, pct in percentages.items() if pct >= 60]

    if len(primary_env) == 1:
        classification = primary_env[0]
        detail = f"{percentages[primary_env[0]]:.0f}% {primary_env[0]}"
    else:
        classification = "multienvironmental"
        detail_parts = [f"{pct:.0f}% {env}" for env, pct in percentages.items() if pct > 0]
        detail = ", ".join(detail_parts) if detail_parts else "no_data"

    return classification, detail

# === Group and process by taxonomy_id ===
final_rows = []

for tax_id, group in df.groupby("taxonomy_id"):
    unique_counts = group[env_cols + ["total_samples"]].drop_duplicates()

    if unique_counts.shape[0] == 1:
        # Case 1: All counts identical
        combined_row = group.iloc[0].copy()
        combined_row["bacdive_id"] = ";".join(sorted(group["bacdive_id"].astype(str).unique()))
        combined_row["description"] = ";".join(sorted(group["description"].astype(str).unique()))
        combined_row["environmental_classification"], combined_row["environment_distribution_details"] = classify_environment(unique_counts.sum())
        combined_row["notes"] = "Merged duplicate entries with identical counts and metadata"
        final_rows.append(combined_row)

    elif group.shape[0] == unique_counts.shape[0]:
        # Case 2: All counts different, sum them
        summed = group[env_cols + ["total_samples"]].sum()
        base_row = group.iloc[0].copy()
        base_row["bacdive_id"] = ";".join(sorted(group["bacdive_id"].astype(str).unique()))
        base_row["description"] = ";".join(sorted(group["description"].astype(str).unique()))
        for col in env_cols + ["total_samples"]:
            base_row[col] = summed[col]
        base_row["environmental_classification"], base_row["environment_distribution_details"] = classify_environment(summed)
        base_row["notes"] = "Summed environmental counts across multiple distinct entries"
        final_rows.append(base_row)

    else:
        # Case 3: Mixed — variations exist
        variation_details = []
        bacdive_per_variation = []

        for i, (_, var_group) in enumerate(group.groupby(env_cols + ["total_samples"])):
            var_row = var_group.iloc[0].copy()
            classification, detail = classify_environment(var_row)
            bac_ids = ";".join(sorted(var_group["bacdive_id"].astype(str).unique()))
            variation_details.append(f"entry variation {i+1} (bacdive_ids: {bac_ids}): {detail}")
            bacdive_per_variation.append(bac_ids)

        # Sum everything
        summed = group[env_cols + ["total_samples"]].sum()
        base_row = group.iloc[0].copy()
        base_row["bacdive_id"] = ";".join(sorted(group["bacdive_id"].astype(str).unique()))
        base_row["description"] = ";".join(sorted(group["description"].astype(str).unique()))
        for col in env_cols + ["total_samples"]:
            base_row[col] = summed[col]

        base_row["environmental_classification"], _ = classify_environment(summed)
        base_row["environment_distribution_details"] = " | ".join(variation_details)
        base_row["notes"] = "Summarized multiple distinct environmental entries with overlapping BacDive sources"
        final_rows.append(base_row)

# === Create final DataFrame and select specific columns ===
final_df = pd.DataFrame(final_rows)

selected_columns = [
    "organism_from_ncb", "taxonomy_id", "strain_id", "bacdive_id", "description",
    "total_samples", "soil_counts", "aquatic_counts", "animal_counts", "plant_counts",
    "environmental_classification", "environment_distribution_details", "notes"
]

final_df = final_df[selected_columns]

# === Save final output ===
output_path = input_path.replace(".csv", "_final_environmental_classification.csv")
final_df.to_csv(output_path, index=False)

# === Print summary ===
unique_tax_ids = final_df["taxonomy_id"].nunique()
print(f"\n✅ Final classification file saved to: {output_path}")
print(f"🔢 Total unique taxonomy_ids (rows in final CSV): {unique_tax_ids}")

classification_counts = final_df.groupby("taxonomy_id")["environmental_classification"].first().value_counts()
print("\n📊 Percentage of each environmental classification (by unique taxonomy_id):")
for cls, count in classification_counts.items():
    pct = (count / unique_tax_ids) * 100
    print(f"  {cls}: {count} ({pct:.2f}%)")



✅ Final classification file saved to: results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found_cleaned_environmental_distribution_final_environmental_classification.csv
🔢 Total unique taxonomy_ids (rows in final CSV): 392

📊 Percentage of each environmental classification (by unique taxonomy_id):
  aquatic: 220 (56.12%)
  multienvironmental: 115 (29.34%)
  soil: 41 (10.46%)
  animal: 13 (3.32%)
  plant: 3 (0.77%)


In [23]:
import pandas as pd

# === Load the final classification file ===
input_path = "results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found_cleaned_environmental_distribution_final_environmental_classification.csv"
df = pd.read_csv(input_path)

# === Filter rows classified as 'animal' ===
animal_df = df[df["environmental_classification"] == "animal"]

# === Get only unique taxonomy_ids (appear exactly once in the full dataset) ===
unique_ids = df["taxonomy_id"].value_counts()
unique_animal_ids = unique_ids[unique_ids == 1].index
animal_unique_df = animal_df[animal_df["taxonomy_id"].isin(unique_animal_ids)]

# === Print the results ===
print(f"\n🦠 Unique taxonomy_ids classified as 'animal': {len(animal_unique_df)} entries\n")
for i, row in animal_unique_df.iterrows():
    print(f"  Taxonomy ID: {row['taxonomy_id']}, Organism: {row['organism_from_ncb']}, BacDive ID(s): {row['bacdive_id']}")



🦠 Unique taxonomy_ids classified as 'animal': 13 entries

  Taxonomy ID: 24, Organism: Shewanella putrefaciens, BacDive ID(s): 14054
  Taxonomy ID: 867, Organism: Ruminobacter amylophilus, BacDive ID(s): 16635
  Taxonomy ID: 82374, Organism: Anaerovibrio lipolyticus, BacDive ID(s): 17117
  Taxonomy ID: 83771, Organism: Succinivibrio dextrinosolvens, BacDive ID(s): 16637
  Taxonomy ID: 200989, Organism: Alkalicoccus saliphilus, BacDive ID(s): 1235
  Taxonomy ID: 333138, Organism: Halalkalibacter okhensis, BacDive ID(s): 1262
  Taxonomy ID: 587909, Organism: Amycolatopsis arida, BacDive ID(s): 13516
  Taxonomy ID: 664784, Organism: Haloechinothrix alba, BacDive ID(s): 13515
  Taxonomy ID: 692418, Organism: Reichenbachiella faecimaris, BacDive ID(s): 5471
  Taxonomy ID: 758803, Organism: Nocardiopsis flavescens, BacDive ID(s): 11241
  Taxonomy ID: 1339210, Organism: Enterovirga rhinocerotis, BacDive ID(s): 140327
  Taxonomy ID: 1545044, Organism: Paracoccus sanguinis, BacDive ID(s): 1313

this is only data stats, no new dataset creation

In [7]:
import pandas as pd

# File paths
ncbi_cleaned_path = "results/environment/vgsc_min100_max500_cdhit_0.90_ncbi_source_info_cleaned.csv"
bacdive_found_path = "results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found_cleaned.csv"

# Load data
df_ncbi = pd.read_csv(ncbi_cleaned_path, dtype=str)
df_bacdive = pd.read_csv(bacdive_found_path, dtype=str)

# Strip whitespace
df_ncbi = df_ncbi.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df_bacdive = df_bacdive.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Columns to check
bacdive_cols = ['Total Samples', 'Isolation Source Categories', 'Isolation Sample Type']
ncbi_cols = ['isolation_source', 'host']

# Environmental terms for BacDive Description column
environment_terms = [
    "soil", "forest", "garden", "desert", "mud", "manure",
    "sediment", "seawater", "water", "lake", "marine", "sea", 
    "coastal", "saline", "hypersaline", "salt", "soda", "pond", 
    "ocean", "intertidal", "tidal", "aquaculture", "saltern", 
    "spring", "hydrothermal", "fluid", "shallow", "plume",
    "human", "blood", "clinical", "stool", "diarrhea", "cow", 
    "animal", "adult", "tissue", "wound", "rumen",
    "plant", "roots", "potato", "solanum", "tuberosum", 
    "maize", "endophytic",
    "sludge", "oil", "metal", "contaminated", "solar", "barn", 
    "industry", "air"
]
env_terms_lower = [term.lower() for term in environment_terms]

# Get unique taxonomy IDs from the NCBI cleaned file
unique_tax_ids = df_ncbi['taxonomy_id'].dropna().unique()

# Tracking results
taxonomy_env_data = {}
column_hits = {
    'Total Samples': 0,
    'Isolation Source Categories': 0,
    'Isolation Sample Type': 0,
    'Description': 0,
    'isolation_source': 0,
    'host': 0
}

# Helper functions
def has_nonempty_value(group, col):
    return col in group.columns and group[col].dropna().apply(lambda x: str(x).strip() != '').any()

def matches_environment_term(desc_series):
    desc_series = desc_series.dropna().str.lower()
    return desc_series.apply(lambda text: any(term in text for term in env_terms_lower)).any()

# Main loop
for tax_id in unique_tax_ids:
    found = False
    bac_rows = df_bacdive[df_bacdive['taxonomy_id'] == tax_id]
    ncb_rows = df_ncbi[df_ncbi['taxonomy_id'] == tax_id]

    # Check BacDive individual columns
    for col in bacdive_cols:
        if has_nonempty_value(bac_rows, col):
            taxonomy_env_data[tax_id] = True
            column_hits[col] += 1
            found = True
            break  # stop after first match

    if not found and 'Description' in bac_rows.columns:
        if matches_environment_term(bac_rows['Description']):
            taxonomy_env_data[tax_id] = True
            column_hits['Description'] += 1
            found = True

    # If still not found, check NCBI columns
    if not found:
        for col in ncbi_cols:
            if has_nonempty_value(ncb_rows, col):
                taxonomy_env_data[tax_id] = True
                column_hits[col] += 1
                found = True
                break

    if not found:
        taxonomy_env_data[tax_id] = False

# Summary
total_ids = len(unique_tax_ids)
with_env = sum(taxonomy_env_data.values())
without_env = total_ids - with_env

# Output
print("=== Environmental Data Summary ===")
print(f"Total unique taxonomy_id in NCBI cleaned file: {total_ids}")
print(f"Unique taxonomy_id with environmental data: {with_env}")
print(f"Unique taxonomy_id without environmental data: {without_env}")
print()
print("=== Breakdown by Source Column ===")
for col, count in column_hits.items():
    print(f"{col}: {count}")


=== Environmental Data Summary ===
Total unique taxonomy_id in NCBI cleaned file: 761
Unique taxonomy_id with environmental data: 661
Unique taxonomy_id without environmental data: 100

=== Breakdown by Source Column ===
Total Samples: 392
Isolation Source Categories: 76
Isolation Sample Type: 61
Description: 1
isolation_source: 124
host: 7


In [9]:
import pandas as pd

# Path to your CSV file
file_path = "results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found_cleaned.csv"

# Columns to check
columns_to_check = [
    "Growth Temp",
    "Optimum Temp",
    "pH Growth",
    "pH Optimum",
    "pH Range",
]

# Read the CSV
df = pd.read_csv(file_path)

# Drop rows where taxonomy_id is missing (just in case)
df = df.dropna(subset=['taxonomy_id'])

# Group by taxonomy_id and aggregate non-null status
grouped = df.groupby('taxonomy_id')[columns_to_check].apply(lambda g: g.notna().any())

# Count how many unique taxonomy_ids have non-null values in each column
print("Number of unique taxonomy_id with at least one value in each column:")
for column in columns_to_check:
    if column in grouped.columns:
        count = grouped[column].sum()
        print(f"{column}: {int(count)}")
    else:
        print(f"{column}: ❌ Column not found in file")


Number of unique taxonomy_id with at least one value in each column:
Growth Temp: 429
Optimum Temp: 293
pH Growth: 229
pH Optimum: 230
pH Range: 190


In [10]:
import pandas as pd

# Define term groups and their merged label
term_groups = {
    'mesophilic': ['mesophile', 'mesophilic'],
    'thermophilic': ['thermophile', 'thermophilic'],
    'psychrophilic': ['psychrophile', 'psychrophilic']
}

# Load the cleaned BacDive file
path = "results/environment/vgsc_min100_max500_cdhit_0.90_bacdive_source_info_found_cleaned.csv"
df = pd.read_csv(path, dtype=str)
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

# Use apply with .str.lower() to avoid deprecated applymap
df_lower = df.apply(lambda col: col.str.lower() if col.dtype == "object" else col)

# Store results
taxonomy_matches = {}
taxonomy_ids = {}

for label, terms in term_groups.items():
    matches = df_lower.apply(
        lambda row: any(any(term in str(cell) for term in terms) for cell in row),
        axis=1
    )
    matching_tax_ids = df.loc[matches, 'taxonomy_id'].dropna().unique()
    taxonomy_matches[label] = len(matching_tax_ids)
    taxonomy_ids[label] = matching_tax_ids  # Save actual IDs

# Print summary
print("=== Unique taxonomy_id matches by temperature preference ===")
for label, count in taxonomy_matches.items():
    print(f"{label}: {count} unique taxonomy_id")

# Print IDs for psychrophilic and thermophilic
print("\n=== taxonomy_id values ===")
for label in ['psychrophilic', 'thermophilic']:
    print(f"\n{label} taxonomy_ids ({taxonomy_matches[label]} total):")
    for tid in taxonomy_ids[label]:
        print(tid)


=== Unique taxonomy_id matches by temperature preference ===
mesophilic: 411 unique taxonomy_id
thermophilic: 22 unique taxonomy_id
psychrophilic: 27 unique taxonomy_id

=== taxonomy_id values ===

psychrophilic taxonomy_ids (27 total):
28108
74033
1121922
1579316
1121923
1076443
86103
999552
245187
195913
619304
1005945
1458307
1129793
1798805
221822
481446
696762
1123010
152297
314281
989403
375451
74031
1886
74348
74349

thermophilic taxonomy_ids (22 total):
1519374
355243
1267768
296745
1390249
933063
1720063
488547
714067
1048
1122252
252514
1642702
201975
996115
747076
360241
696763
1003
103836
35622
77097


trying to isolate data 

just testing optimizztion shit you can ignore this

In [51]:
import pandas as pd

# Path to your CSV file
file_path = 'results/environment/vgsc-id90_min100_max500_bacdive_source_info_found_cleaned.csv'

# Load the CSV into a DataFrame
df = pd.read_csv(file_path)

# Specify the row indices you want to print
# For example, rows 2, 5, and 10 (indexing starts at 0)
rows_to_print = [524, 523]

# Print the selected rows
print(df.loc[rows_to_print])


     organism_from_ncb  taxonomy_id strain_id  BacDive-ID  \
524  Ruegeria pomeroyi        89184       NaN       13783   
523  Ruegeria pomeroyi        89184       NaN       13784   

                                           Description  \
524  Ruegeria pomeroyi DSS-3 is a mesophilic, Gram-...   
523  Ruegeria pomeroyi DSS-10 is a mesophilic bacte...   

                                              Keywords Growth Temp  \
524  genome sequence, 16S sequence, Bacteria, mesop...         NaN   
523                               Bacteria, mesophilic         NaN   

    Optimum Temp pH Growth pH Optimum  ... Isolation Sample Type  \
524          NaN       NaN        NaN  ...              seawater   
523          NaN       NaN        NaN  ...              seawater   

    Isolation Geographic Location Isolation Country Isolation Continent  \
524              coast of Georgia               USA       North America   
523               coastal Georgia               USA       North America   


In [56]:
import pandas as pd
from collections import Counter
import re

# Path to your CSV file
file_path = 'results/environment/vgsc-id90_min100_max500_bacdive_source_info_found.csv'

# Load the CSV into a DataFrame
df = pd.read_csv(file_path)

# Combine all text from the 'Description' column into one large string
all_text = ' '.join(df['Description'].dropna().astype(str))

# Clean and split the text into words (lowercase, remove non-alphabetic characters)
words = re.findall(r'\b[a-zA-Z]+\b', all_text.lower())

# Count the frequency of each word
word_counts = Counter(words)

# Convert to a list of (word, count) tuples sorted by frequency (most common first)
most_common_words = word_counts.most_common()

# Print the list
for word, count in most_common_words:
    print(f"{word}: {count}")


is: 1068
bacterium: 1029
a: 801
from: 761
that: 702
was: 689
isolated: 689
mesophilic: 636
of: 484
gram: 427
the: 416
an: 390
family: 368
negative: 363
streptomyces: 341
aerobe: 328
streptomycetaceae: 275
avidinii: 229
spore: 154
forming: 154
soil: 134
dsm: 124
and: 97
sediment: 92
seawater: 82
water: 78
positive: 63
ccug: 61
shewanella: 61
putrefaciens: 61
sea: 59
anaerobe: 57
jcm: 55
albidoflavus: 54
lake: 52
marine: 51
obligate: 46
motile: 46
shaped: 44
forms: 44
colonies: 44
circular: 42
pathogen: 39
surface: 37
rod: 35
plant: 35
pseudomonas: 35
sample: 34
facultative: 34
builds: 32
aerial: 32
mycelium: 32
shewanellaceae: 30
scabiei: 29
deep: 27
oleovorans: 27
psychrophilic: 25
human: 24
micromonospora: 24
bacillus: 23
sulfitobacter: 23
hypersaline: 22
produces: 22
amyloliquefaciens: 22
antibiotic: 20
compounds: 20
marinobacter: 20
halomonas: 19
alkalihalophilus: 18
pseudofirmus: 18
in: 17
saline: 17
at: 16
coastal: 16
sp: 15
flat: 15
with: 15
sediments: 15
m: 14
tidal: 14
cip: 14


In [58]:
import pandas as pd
from collections import Counter
import re

# Path to your CSV file
file_path = 'results/environment/vgsc-id90_min100_max500_bacdive_source_info_found.csv'

# Load the CSV
df = pd.read_csv(file_path)

# Extract and clean all words from the 'Description' column
all_text = ' '.join(df['Description'].dropna().astype(str))
words = re.findall(r'\b[a-zA-Z]+\b', all_text.lower())
word_counts = Counter(words)

# Get the N most common words (you can change N if needed)
N = 200  # top 100 most common words
most_common_words = word_counts.most_common(N)

# Format as a paragraph
formatted = ', '.join([f"{word} ({count})" for word, count in most_common_words])

# Print the result
print(formatted)


is (1068), bacterium (1029), a (801), from (761), that (702), was (689), isolated (689), mesophilic (636), of (484), gram (427), the (416), an (390), family (368), negative (363), streptomyces (341), aerobe (328), streptomycetaceae (275), avidinii (229), spore (154), forming (154), soil (134), dsm (124), and (97), sediment (92), seawater (82), water (78), positive (63), ccug (61), shewanella (61), putrefaciens (61), sea (59), anaerobe (57), jcm (55), albidoflavus (54), lake (52), marine (51), obligate (46), motile (46), shaped (44), forms (44), colonies (44), circular (42), pathogen (39), surface (37), rod (35), plant (35), pseudomonas (35), sample (34), facultative (34), builds (32), aerial (32), mycelium (32), shewanellaceae (30), scabiei (29), deep (27), oleovorans (27), psychrophilic (25), human (24), micromonospora (24), bacillus (23), sulfitobacter (23), hypersaline (22), produces (22), amyloliquefaciens (22), antibiotic (20), compounds (20), marinobacter (20), halomonas (19), al

In [68]:
import bacdive
import json  # Optional, for pretty printing

# Authenticate
client = bacdive.BacdiveClient('username', 'password') #insert username and password

# Set optional search type (not strictly needed for ID-based search)
client.setSearchType('exact')

# Define BacDive IDs to search for
bacdive_ids = [4301]  # You can change this list to your desired IDs

# Perform the search
client.search(id=bacdive_ids)

# Retrieve and print results
for strain in client.retrieve():
    # Pretty print the full strain entry as JSON
    print(json.dumps(strain, indent=2, ensure_ascii=False))


-- Authentication successful --
{
  "General": {
    "@ref": 7395,
    "BacDive-ID": 4301,
    "DSM-Number": 18102,
    "keywords": [
      "genome sequence",
      "16S sequence",
      "Bacteria",
      "facultative aerobe",
      "mesophilic",
      "Gram-negative",
      "motile",
      "rod-shaped"
    ],
    "description": "Aquisalimonas asiatica DSM 18102 is a facultative aerobe, mesophilic, Gram-negative bacterium that was isolated from water samples from the alkaline, saline Lake Chagannor .",
    "NCBI tax id": {
      "NCBI tax id": 406100,
      "Matching level": "species"
    },
    "strain history": [
      {
        "@ref": 7395,
        "history": "<- A. Ventosa, Univ. Seville, Dept. Microbiol. Parasitol., Spain; CG12 <- I. J. Carrasco et al."
      },
      {
        "@ref": 116524,
        "history": "CIP <- 2007, CECT <- 2006, A. Ventosa, Sevilla Univ., Sevilla, Spain: strain CG12"
      }
    ],
    "doi": "10.13145/bacdive4301.20250331.9.3"
  },
  "Name and taxonom

In [72]:
#this is an improve version of the code 
import pandas as pd
import os
import sys
import io
from bacdive import BacdiveClient
from tqdm import tqdm

# === Toggle Test Mode ===
TEST_MODE = True  # Set to False to process the full dataset

# Manual BacDive IDs to test
manual_bacdive_ids = [4301]

# Load input CSV
input_csv = "results/environment/vgsc-id90_min100_max500_ncbi_source_info_cleaned.csv"
df = pd.read_csv(input_csv)

# Extract base name
base_filename = os.path.basename(input_csv)
base_name = base_filename.split('_ncbi')[0]

# Output filenames (add "_test" if test mode is on)
suffix = "_test" if TEST_MODE else ""
output_file = f"results/environment/{base_name}_bacdive_source_info_found{suffix}.csv"
not_found_file = f"results/environment/{base_name}_bacdive_source_info_missing{suffix}.csv"

# Initialize BacDive client
client = BacdiveClient('username', 'password') #insert username and password

# Results containers
results = []
not_found = []
processed_taxonomy_ids = set()

# === Silent wrapper to suppress stdout ===
def silent_search(func, *args, **kwargs):
    old_stdout = sys.stdout
    sys.stdout = io.StringIO()
    try:
        return func(*args, **kwargs)
    finally:
        sys.stdout = old_stdout

# === Step 1: Retrieve manual BacDive IDs ===
if TEST_MODE and manual_bacdive_ids:
    print(f"🔬 Fetching manual BacDive ID(s): {manual_bacdive_ids}")
    client.setSearchType('exact')
    silent_search(client.search, id=manual_bacdive_ids)
    for strain_data in client.retrieve():
        bacdive_id = strain_data.get('General', {}).get('BacDive-ID')
        if not bacdive_id:
            continue

        general = strain_data.get('General', {})
        description = general.get('description', '')
        keywords = general.get('keywords', '')
        if isinstance(keywords, list):
            keywords = ", ".join(keywords)

        isolation = strain_data.get('Isolation, sampling and environmental information', {})
        iso_cat = isolation.get('isolation source categories', [])
        iso_cat_str = ''
        if isinstance(iso_cat, list):
            category_values = []
            for cat_dict in iso_cat:
                if isinstance(cat_dict, dict):
                    category_values.extend([v for v in cat_dict.values() if v])
            iso_cat_str = ", ".join(sorted(set(category_values)))

        results.append({
            "organism_from_ncb": '',
            "taxonomy_id": '',
            "strain_id": '',
            "BacDive-ID": bacdive_id,
            "Description": description,
            "Keywords": keywords,
            "Growth Temp": '',
            "Optimum Temp": '',
            "pH Growth": '',
            "pH Optimum": '',
            "pH Range": '',
            "Isolation Sample Type": '',
            "Isolation Geographic Location": '',
            "Isolation Country": '',
            "Isolation Continent": '',
            "Isolation Source Categories": iso_cat_str,
            "Total Samples": '',
            "Soil Counts": '',
            "Aquatic Counts": '',
            "Animal Counts": '',
            "Plant Counts": ''
        })

# === Step 2: Process first 10 rows from CSV (test) or full (normal) ===
rows_to_process = df.head(10) if TEST_MODE else df

# Loop through rows
for _, row in tqdm(rows_to_process.iterrows(), total=len(rows_to_process), desc="Fetching BacDive entries"):
    strain_id = str(row.get("strain_id", "")).strip()
    organism = str(row.get("organism_from_ncb", "")).strip()
    tax_id = str(row.get("taxonomy_id", "")).strip()

    if tax_id in processed_taxonomy_ids:
        continue

    data_found = False
    bacdive_ids_added = set()

    count = 0
    if strain_id:
        client.setSearchType('exact')
        count = silent_search(client.search, culturecolno=strain_id)

    if count == 0 and organism:
        client.setSearchType('startswith')
        count = silent_search(client.search, taxonomy=organism)

    if count == 0:
        not_found.append({
            "organism_from_ncb": organism,
            "taxonomy_id": tax_id,
            "strain_id": strain_id
        })
        processed_taxonomy_ids.add(tax_id)
        continue

    for strain_data in client.retrieve():
        bacdive_id = strain_data.get('General', {}).get('BacDive-ID')
        if not bacdive_id or bacdive_id in bacdive_ids_added:
            continue

        bacdive_ids_added.add(bacdive_id)
        data_found = True

        general = strain_data.get('General', {})
        description = general.get('description', '')
        keywords = general.get('keywords', '')
        if isinstance(keywords, list):
            keywords = ", ".join(keywords)

        # Temperature
        temps = strain_data.get('Culture and growth conditions', {}).get('culture temp', [])
        growth_temps = set()
        optimum_temps = set()
        for entry in temps:
            if isinstance(entry, dict):
                if entry.get('type') == 'growth':
                    growth_temps.add(entry.get('temperature', ''))
                elif entry.get('type') == 'optimum':
                    optimum_temps.add(entry.get('temperature', ''))

        # pH
        ph_entries = strain_data.get('Culture and growth conditions', {}).get('culture pH', [])
        ph_growth = ''
        ph_optimum = ''
        ph_range = ''
        for entry in ph_entries:
            if isinstance(entry, dict):
                if entry.get('type') == 'growth':
                    ph_growth = entry.get('pH', '')
                    ph_range = entry.get('PH range', '')
                elif entry.get('type') == 'optimum':
                    ph_optimum = entry.get('pH', '')

        # Isolation
        isolation = strain_data.get('Isolation, sampling and environmental information', {})
        iso_entries = isolation.get('isolation', [])
        sample_type = geo_location = country = continent = ''
        if isinstance(iso_entries, list) and iso_entries:
            first = iso_entries[0]
            sample_type = first.get('sample type', '')
            geo_location = first.get('geographic location', '')
            country = first.get('country', '')
            continent = first.get('continent', '')
        elif isinstance(iso_entries, dict):
            sample_type = iso_entries.get('sample type', '')
            geo_location = iso_entries.get('geographic location', '')
            country = iso_entries.get('country', '')
            continent = iso_entries.get('continent', '')

        # Source categories
        iso_cat = isolation.get('isolation source categories', [])
        iso_cat_str = ''
        if isinstance(iso_cat, list):
            category_values = []
            for cat_dict in iso_cat:
                if isinstance(cat_dict, dict):
                    category_values.extend([v for v in cat_dict.values() if v])
            iso_cat_str = ", ".join(sorted(set(category_values)))

        taxonmaps = isolation.get('taxonmaps', {})
        total_samples = taxonmaps.get('Total samples', '')
        soil_counts = taxonmaps.get('soil counts', '')
        aquatic_counts = taxonmaps.get('aquatic counts', '')
        animal_counts = taxonmaps.get('animal counts', '')
        plant_counts = taxonmaps.get('plant counts', '')

        results.append({
            "organism_from_ncb": organism,
            "taxonomy_id": tax_id,
            "strain_id": strain_id,
            "BacDive-ID": bacdive_id,
            "Description": description,
            "Keywords": keywords,
            "Growth Temp": ", ".join(sorted(growth_temps)),
            "Optimum Temp": ", ".join(sorted(optimum_temps)),
            "pH Growth": ph_growth,
            "pH Optimum": ph_optimum,
            "pH Range": ph_range,
            "Isolation Sample Type": sample_type,
            "Isolation Geographic Location": geo_location,
            "Isolation Country": country,
            "Isolation Continent": continent,
            "Isolation Source Categories": iso_cat_str,
            "Total Samples": total_samples,
            "Soil Counts": soil_counts,
            "Aquatic Counts": aquatic_counts,
            "Animal Counts": animal_counts,
            "Plant Counts": plant_counts
        })

    processed_taxonomy_ids.add(tax_id)

    if not data_found:
        not_found.append({
            "organism_from_ncb": organism,
            "taxonomy_id": tax_id,
            "strain_id": strain_id
        })

# Save CSVs
pd.DataFrame(results).to_csv(output_file, index=False)
pd.DataFrame(not_found).to_csv(not_found_file, index=False)

print(f"\n✅ Saved: {output_file}")
print(f"❌ Missing: {not_found_file}")


-- Authentication successful --
🔬 Fetching manual BacDive ID(s): [4301]


Fetching BacDive entries: 100%|████████████████████████████████████████████████████████| 10/10 [00:11<00:00,  1.14s/it]


✅ Saved: results/environment/vgsc-id90_min100_max500_bacdive_source_info_found_test.csv
❌ Missing: results/environment/vgsc-id90_min100_max500_bacdive_source_info_missing_test.csv


In [ ]:
the code above is better, implement after meeting